# 6.4 · DBSCAN / Density-Based Spatial Clustering

> **课程定位 / Where this fits**
> 第 4 课，**Part 6 · 无监督学习**。
> Lesson 4, **Part 6 · Unsupervised Learning**.
>
> K-Means(6.1)、层次聚类(6.3)都假设簇是团状/球形，还得（间接）指定簇数，也不会处理噪声。**DBSCAN** 换了个范式：簇 = **高密度区域**，由密度连通来定义。它能找**任意形状**的簇、**自动识别噪声/离群点**、**不需预先指定簇数**。月牙、环形这些 K-Means 的死穴，DBSCAN 轻松搞定。
> K-Means (6.1) and hierarchical (6.3) assume blob/spherical clusters, (indirectly) need a cluster count, and ignore noise. **DBSCAN** flips the paradigm: a cluster = a **high-density region**, defined by density connectivity. It finds **arbitrary shapes**, **auto-detects noise/outliers**, and **needs no preset cluster count**. Moons and rings — K-Means's nemeses — are easy for DBSCAN.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - eps ($\varepsilon$) —— 邻域半径 / neighborhood radius
> - min_samples —— 成为核心点所需的邻居数 / neighbors required to be a core point
> - 标签 −1 —— 噪声点 / noise points are labeled −1

> 💡 **面试相关 / Interview-relevant**
> - "DBSCAN 的核心/边界/噪声点定义"（★★★★★）
> - "eps 和 min_samples 怎么调（k-distance 图）"（★★★★★）
> - "DBSCAN 优缺点 vs K-Means"（★★★★★）
> - "DBSCAN 为什么对不同密度的簇会失败"（★★★★，引出 HDBSCAN）

---

## 学习目标 / Learning Objectives

1. 理解核心点/边界点/噪声点 + 密度可达。
   Understand core/border/noise points + density-reachability.
2. **从零**实现 DBSCAN（看清密度连通）。
   Implement DBSCAN **from scratch**.
3. 用 k-distance 肘部调 **eps / min_samples**。
   Tune **eps / min_samples** via the k-distance elbow.
4. 体会任意形状 + 噪声识别的威力。
   Appreciate arbitrary shapes + noise detection.
5. 理解不同密度簇的局限（为 6.5 HDBSCAN 铺垫）。
   Understand the varying-density limitation (motivating HDBSCAN, 6.5).

## 目录 / TOC
1. [先建直觉：扎堆才算簇](#1)
2. [核心/边界/噪声 + 密度可达 ⭐](#2)
3. [🌙 数据：moons + 从零实现 ⭐](#3)
4. [eps/min_samples 调参 ⭐](#4)
5. [任意形状 + 噪声 vs K-Means](#5)
6. [不同密度的局限 ⭐](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 先建直觉：扎堆才算簇 / Intuition First

K-Means 问"离哪个中心近"，DBSCAN 问的是完全不同的问题：**"这个点周围挤不挤？"** 如果一个点周围一小圈里挤了足够多的邻居，它就处在"稠密区"，和它连成一片的稠密区域就构成一个簇；而孤零零待在稀疏处的点，就被判为**噪声**。
K-Means asks "which center is nearest"; DBSCAN asks a totally different question: **"is it crowded around this point?"** If a point has enough neighbors within a small radius, it sits in a "dense region"; connected dense regions form a cluster, while points stranded in sparse areas are flagged as **noise**.

因为簇是顺着"稠密"一路蔓延出来的，它可以长成**任意形状**（月牙、环、蛇形都行），完全不像 K-Means 只能画圆。而且簇的数量是**自然涌现**的，不用我们指定。
Because clusters grow by following density, they can take **any shape** (crescents, rings, snakes) — unlike K-Means's circles. And the number of clusters **emerges naturally**, no need to specify it.


<a id="2"></a>
## 2. 核心/边界/噪声 + 密度可达 ⭐ / Core, Border, Noise

DBSCAN 有两个超参：**eps（$\varepsilon$，邻域半径）** 和 **min_samples（成为核心点所需的邻居数）**。据此把每个点分成三类：
DBSCAN has two hyperparameters: **eps (radius)** and **min_samples (neighbors needed to be a core point)**. Each point falls into one of three types:
- **核心点 core**：它的 $\varepsilon$ 邻域内至少有 min_samples 个点（含自己）→ 处在稠密区。
  At least min_samples points within its $\varepsilon$-neighborhood (including itself) → in a dense region.
- **边界点 border**：自己不够核心，但落在某个核心点的 $\varepsilon$ 邻域内。
  Not core itself, but lies within some core point's $\varepsilon$-neighborhood.
- **噪声点 noise**：既非核心也非边界 → 离群，标签 −1。
  Neither core nor border → an outlier, labeled −1.

**簇怎么形成**：从一个核心点出发，把所有**密度可达**的点（经由一串核心点的 $\varepsilon$ 邻域链相连）纳入同一个簇。簇因此能沿着密度"长"成任意形状。簇数由数据密度自然决定，无需指定。
**How a cluster forms:** start from a core point and absorb all **density-reachable** points (linked through a chain of core points' $\varepsilon$-neighborhoods) into one cluster. Clusters thus grow along density into any shape, and their count emerges from the data.


<a id="3"></a>
## 3. 数据：moons + 从零实现 ⭐ / Data & From Scratch

**make_moons**：两个交错的半月形，线性不可分、非球形——这是 DBSCAN 的主场、K-Means 的噩梦。我们再撒几个离群点测试噪声识别。
**make_moons**: two interlocking crescents, not linearly separable, non-spherical — DBSCAN's home turf and K-Means's nightmare. We add a few outliers to test noise detection.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons
sns.set_theme(style="whitegrid")

X, _ = make_moons(300, noise=0.06, random_state=0)
rng = np.random.default_rng(0)
X = np.vstack([X, rng.uniform(-1.5, 2.5, (12, 2))])   # 额外撒 12 个随机离群点当噪声
print(f"moons + 噪声 noise: {X.shape}")

def dbscan_scratch(X, eps, min_samples):
    n = len(X)
    labels = np.full(n, -1)               # 所有点先标 -1(噪声), 后续再归簇
    cluster = 0                           # 当前簇编号
    # 距离矩阵: D[i,j] = 点 i 到点 j 的欧氏距离 (用广播一次算完)
    D = np.sqrt(((X[:,None,:] - X[None,:,:])**2).sum(-1))
    visited = np.zeros(n, bool)           # 标记每个点是否已被处理过
    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        nbrs = np.where(D[i] <= eps)[0]   # 点 i 的 ε 邻域内所有点的下标
        if len(nbrs) < min_samples:       # 邻居不够 → 暂判噪声(以后可能被收编为边界点)
            continue
        labels[i] = cluster               # i 是核心点, 开一个新簇
        seeds = list(nbrs)                # 待扩张的种子队列(从 i 的邻居开始)
        k = 0
        while k < len(seeds):             # 广度优先地把整片稠密区收进同一簇
            j = seeds[k]; k += 1
            if labels[j] == -1:           # 之前是噪声 → 现在变成这个簇的边界点
                labels[j] = cluster
            if not visited[j]:
                visited[j] = True
                jn = np.where(D[j] <= eps)[0]
                if len(jn) >= min_samples:   # j 也是核心点 → 把它的邻居也加入扩张队列
                    seeds += list(jn)
        cluster += 1
    return labels

lab_scratch = dbscan_scratch(X, eps=0.2, min_samples=5)
from sklearn.cluster import DBSCAN
lab_sk = DBSCAN(eps=0.2, min_samples=5).fit_predict(X)
# 统计簇数(去掉噪声标签 -1)和噪声点数
print(f"从零 DBSCAN: {len(set(lab_scratch))-(1 if -1 in lab_scratch else 0)} 簇, {(lab_scratch==-1).sum()} 噪声")
print(f"sklearn   : {len(set(lab_sk))-(1 if -1 in lab_sk else 0)} 簇, {(lab_sk==-1).sum()} 噪声")
from sklearn.metrics import adjusted_rand_score
print(f"两者一致性 agreement ARI = {adjusted_rand_score(lab_scratch, lab_sk):.3f}")


<a id="4"></a>
## 4. eps/min_samples 调参 ⭐ / Tuning

DBSCAN 对 **eps** 很敏感。常用调法：画 **k-distance 图**——算每个点到它第 k(=min_samples) 近邻的距离，从小到大排序画出来。曲线的**肘部（陡然上升处）**就是好的 eps：肘部以下是"簇内距离"，以上是"跨簇/噪声距离"。
DBSCAN is sensitive to **eps**. A common method: the **k-distance plot** — for each point compute the distance to its k-th (=min_samples) nearest neighbor, sort ascending, and plot. The **elbow (sharp rise)** is a good eps: below it are within-cluster distances, above it are cross-cluster/noise distances.


In [ ]:
from sklearn.neighbors import NearestNeighbors
k = 5
nn = NearestNeighbors(n_neighbors=k).fit(X)
# kneighbors 返回(距离, 下标); [0] 取距离, [:, -1] 取每个点到第 k 近邻的距离
kdist = np.sort(nn.kneighbors(X)[0][:, -1])    # 升序排列, 便于看肘部
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(kdist)
ax.axhline(0.2, color="r", ls="--", label="肘部 elbow eps≈0.2")
ax.set_xlabel("点(按 k-distance 升序) points sorted by k-distance"); ax.set_ylabel(f"到第{k}近邻距离 dist to {k}-th NN"); ax.legend()
ax.set_title("k-distance 图: 肘部(陡升处)即合适 eps / elbow = good eps")
plt.tight_layout(); plt.show()
print("eps 取肘部值; min_samples 经验 ≈ 2×维度, 噪声多则调大 / eps from elbow, min_samples≈2×dim")


<a id="5"></a>
## 5. 任意形状 + 噪声 vs K-Means / Arbitrary Shapes & Noise

直接对比：同样的月牙+噪声数据，DBSCAN 抓住两条弯月并标出离群点，K-Means 只会球形切分、还把噪声硬塞进簇。
A direct comparison: on the same moons+noise data, DBSCAN captures both crescents and flags outliers, while K-Means can only cut spherically and forces noise into clusters.


In [ ]:
from sklearn.cluster import KMeans
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
mask = lab_sk == -1                        # 被判为噪声的点
axes[0].scatter(X[~mask,0], X[~mask,1], c=lab_sk[~mask], cmap="coolwarm", s=18)
axes[0].scatter(X[mask,0], X[mask,1], c="k", marker="x", s=40, label="噪声 noise(-1)")
axes[0].set_title("DBSCAN: 两条月牙 + 自动标出噪声 / two moons + noise flagged"); axes[0].legend()

km = KMeans(2, n_init=10, random_state=0).fit_predict(X)
axes[1].scatter(X[:,0], X[:,1], c=km, cmap="coolwarm", s=18)
axes[1].set_title("K-Means: 强行球形切分, 切错且无噪声概念 / spherical cut, no noise concept")
plt.tight_layout(); plt.show()
print("DBSCAN 抓住非球形簇并隔离离群点; K-Means 只能球形划分、把噪声硬塞进簇")


<a id="6"></a>
## 6. 不同密度的局限 ⭐ / The Varying-Density Limitation

DBSCAN 用**单一的全局 eps**。如果数据里有的簇稠密、有的簇稀疏，**一个 eps 无法同时合适**：eps 小 → 稀疏簇被打散成噪声；eps 大 → 稠密的小簇被合并到一起。这是 DBSCAN 的根本短板，也正是 **HDBSCAN(6.5)** 要解决的——它用层次化的密度，自动适应不同密度。
DBSCAN uses **one global eps**. If some clusters are dense and others sparse, **no single eps works for all**: small eps → sparse clusters shatter into noise; large eps → dense small clusters merge. This is DBSCAN's core weakness, exactly what **HDBSCAN (6.5)** fixes — using hierarchical density to adapt to varying density.


In [ ]:
from sklearn.cluster import DBSCAN
# 造两个密度差异很大的簇: 一个紧凑(std=0.3), 一个松散(std=1.0)
rng = np.random.default_rng(1)
dense = rng.normal([0,0], 0.3, (200,2))    # 稠密簇
sparse = rng.normal([3,3], 1.0, (200,2))   # 稀疏簇
Xd = np.vstack([dense, sparse])
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, eps in zip(axes, [0.3, 0.6, 1.0]):     # 试三个 eps, 看没有一个能两全
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xd)
    nnoise = (lab==-1).sum(); nclu = len(set(lab))-(1 if -1 in lab else 0)
    m = lab==-1
    ax.scatter(Xd[~m,0], Xd[~m,1], c=lab[~m], cmap="tab10", s=12)
    ax.scatter(Xd[m,0], Xd[m,1], c="k", marker="x", s=20)
    ax.set_title(f"eps={eps}: {nclu}簇 clusters, {nnoise}噪声 noise")
plt.suptitle("单一全局 eps 难兼顾不同密度: 小eps 拆散稀疏簇, 大eps 合并稠密簇 / one eps can't fit both")
plt.tight_layout(); plt.show()
print("没有一个 eps 能同时正确处理稠密簇和稀疏簇 → HDBSCAN(6.5)用层次密度解决")


<a id="7"></a>
## 7. 小结 / Summary

```
DBSCAN: 簇=密度连通的高密区域; 两超参 eps(邻域半径) + min_samples(成核阈值)
点分三类: 核心(邻域≥min_samples) / 边界(在核心邻域内) / 噪声(-1)
密度可达: 从核心点经核心链扩张 → 任意形状簇, 自动识别噪声, 不预设簇数
调 eps: k-distance 图的肘部; min_samples ≈ 2×维度
局限: 单一全局 eps 无法兼顾不同密度的簇 → HDBSCAN(6.5)
```

### 💡 面试速查 / Interview cheat-sheet
1. **核心/边界/噪声**三类点；簇由密度可达连通成任意形状。
   Core/border/noise; clusters grow by density-reachability into any shape.
2. **eps**（k-distance 肘部）+ **min_samples**（≈2×维度）。
   eps (k-distance elbow) + min_samples (≈2×dim).
3. **优点**：任意形状、自动噪声、不预设 K；**缺点**：对 eps 敏感、怕不同密度、高维退化。
   Pros: any shape, auto noise, no K; cons: eps-sensitive, varying density, high-dim decay.
4. **不同密度失败** → HDBSCAN。
   Fails on varying density → HDBSCAN.
5. **vs K-Means**：DBSCAN 非球形+噪声，K-Means 球形+快+可扩展。
   vs K-Means: DBSCAN non-spherical+noise, K-Means spherical+fast+scalable.

### 下一节 / Next
**6.5 HDBSCAN**——把 DBSCAN 层次化，在所有密度尺度上构建层次并自动选出最稳定的簇，不再需要手调单一 eps。
**6.5 HDBSCAN** — makes DBSCAN hierarchical across all density scales and auto-selects the most stable clusters, removing the single-eps tuning.
